# House Price Prediction Advanced Competition

Import libraries and load dataset.

In [36]:
import numpy as np
import pandas as pd

In [37]:
df = pd.read_csv("train.csv")
df.head(4)

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000


In [38]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   str    
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   str    
 6   Alley          91 non-null     str    
 7   LotShape       1460 non-null   str    
 8   LandContour    1460 non-null   str    
 9   Utilities      1460 non-null   str    
 10  LotConfig      1460 non-null   str    
 11  LandSlope      1460 non-null   str    
 12  Neighborhood   1460 non-null   str    
 13  Condition1     1460 non-null   str    
 14  Condition2     1460 non-null   str    
 15  BldgType       1460 non-null   str    
 16  HouseStyle     1460 non-null   str    
 17  OverallQual    1460 non-null   int64  
 18  OverallCond    1460

In [39]:
# sort by unique values
df.describe(include="object").sort_values(by="unique", axis=1)

/var/folders/bc/1q84krsx5vgcsc01l1rp77_r0000gn/T/ipykernel_13803/1648093361.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.describe(include="object").sort_values(by="unique", axis=1)


,Street,Alley,CentralAir,Utilities,MasVnrType,LandSlope,PoolQC,PavedDrive,GarageFinish,BsmtQual,...,SaleCondition,Functional,RoofMatl,HouseStyle,Condition2,SaleType,Condition1,Exterior1st,Exterior2nd,Neighborhood
count,1460,91,1460,1460,588,1460,7,1460,1379,1423,...,1460,1460,1460,1460,1460,1460,1460,1460,1460,1460
unique,2,2,2,2,3,3,3,3,3,4,...,6,7,8,8,8,9,9,15,16,25
top,Pave,Grvl,Y,AllPub,BrkFace,Gtl,Gd,Y,Unf,TA,...,Normal,Typ,CompShg,1Story,Norm,WD,Norm,VinylSd,VinylSd,NAmes
freq,1454,50,1365,1459,445,1382,3,1340,605,649,...,1198,1360,1434,726,1445,1267,1260,515,504,225


In [40]:
sorted_counts = pd.DataFrame({"counts": df.isna().sum(), "types": df.dtypes})
sorted_counts = sorted_counts.sort_values(by="counts", ascending=False)
sorted_counts = sorted_counts[sorted_counts["counts"] > 0]
sorted_counts

,counts,types
PoolQC,1453,str
MiscFeature,1406,str
Alley,1369,str
Fence,1179,str
MasVnrType,872,str
FireplaceQu,690,str
LotFrontage,259,float64
GarageYrBlt,81,float64
GarageCond,81,str
GarageType,81,str


In [41]:
def missing_value_imp(column) -> None:
    missing_df = df[df[column].isna()]
    has_df = df[df[column].notna()]
    
    missing_mean_price = missing_df["SalePrice"].mean()
    has_mean_price = has_df["SalePrice"].mean()

    print(f"Missing Mean Price: {missing_mean_price:.2f}")
    print(f"Present Mean Price: {has_mean_price:.2f}")

In [42]:
def class_difs(column) -> None:
    mean_prices = pd.DataFrame()
    for val in df[column].unique():
        if (pd.isna(val)):
            selected_df = df[df[column].isna()]
        else:
            selected_df = df[df[column] == val]

        mean_prices.loc[val, "Mean"] = round(selected_df["SalePrice"].mean(), 2)
        mean_prices.loc[val, "Count"] = len(selected_df)
    
    std = mean_prices["Mean"].std()
    
    return (mean_prices, std)

In [43]:
pairs = []

for col in df.columns:
    if df[col].dtype == "str":
        mean_prices, std = class_difs(col)
        pairs.append((col, std))

pairs.sort(key=lambda x: x[1], reverse=True)
pairs

[('PoolQC', np.float64(146067.73957720975)),
 ('ExterQual', np.float64(121669.39047360662)),
 ('KitchenQual', np.float64(98569.61992362127)),
 ('BsmtQual', np.float64(91276.08970090124)),
 ('Condition2', np.float64(84325.7167999827)),
 ('RoofMatl', np.float64(79783.31889417898)),
 ('FireplaceQu', np.float64(76321.93838338323)),
 ('Neighborhood', np.float64(66725.19765488396)),
 ('GarageFinish', np.float64(60945.93204776099)),
 ('GarageQual', np.float64(60885.59655677044)),
 ('BsmtCond', np.float64(60377.57988927885)),
 ('Exterior1st', np.float64(57697.23739692198)),
 ('CentralAir', np.float64(57220.94749551774)),
 ('SaleType', np.float64(57043.87123673215)),
 ('MiscFeature', np.float64(56291.14423319205)),
 ('SaleCondition', np.float64(56214.329416193665)),
 ('MSZoning', np.float64(55553.077879077675)),
 ('Exterior2nd', np.float64(55551.54477253886)),
 ('BsmtExposure', np.float64(55043.6848382299)),
 ('MasVnrType', np.float64(54414.12423263466)),
 ('ExterCond', np.float64(54186.4891529

In [44]:
def convertColsToHas(cols: list[str]) -> pd.DataFrame:
    for col in cols:
        df["Has"+col] = df[col].notna().map({False: 0, True: 1})
   
    return df

In [45]:
convertColsToHas(cols=["Alley", "Fence", "MasVnrType"])

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice,HasAlley,HasFence,HasMasVnrType
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,NaN,0,2,2008,WD,Normal,208500,0,0,1
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,NaN,0,5,2007,WD,Normal,181500,0,0,0
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,NaN,0,9,2008,WD,Normal,223500,0,0,1
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,NaN,0,2,2006,WD,Abnorml,140000,0,0,0
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,NaN,0,12,2008,WD,Normal,250000,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1455,1456,60,RL,62.0,7917,Pave,NaN,Reg,Lvl,AllPub,...,NaN,0,8,2007,WD,Normal,175000,0,0,0
1456,1457,20,RL,85.0,13175,Pave,NaN,Reg,Lvl,AllPub,...,NaN,0,2,2010,WD,Normal,210000,0,1,1
1457,1458,70,RL,66.0,9042,Pave,NaN,Reg,Lvl,AllPub,...,Shed,2500,5,2010,WD,Normal,266500,0,1,0
1458,1459,20,RL,68.0,9717,Pave,NaN,Reg,Lvl,AllPub,...,NaN,0,4,2010,WD,Normal,142125,0,0,0


In [46]:
df["Electrical"] = df["Electrical"].fillna(df["Electrical"].mode().iloc[0])
df["MasVnrArea"] = df["MasVnrArea"].fillna(df["MasVnrArea"].median())
df["LotFrontage"] = df["LotFrontage"].fillna(df["LotFrontage"].median())
df["GarageYrBlt"] = df["GarageYrBlt"].fillna(df["GarageYrBlt"].median())

In [47]:
df = df.drop(columns=["PoolQC", "MiscFeature", "Alley", "Fence", "MasVnrType", "SalePrice", "Id"])

Create ordinal encodings for quality variables.

In [48]:
df["OverallQual"]

0       7
1       6
2       7
3       7
4       8
       ..
1455    6
1456    6
1457    7
1458    5
1459    5
Name: OverallQual, Length: 1460, dtype: int64

In [49]:
quality_ordinal = {pd.NA: 0, "NA": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5}

for col in df.columns:
    if "Qu" in col:
        df[col] = df[col].replace(quality_ordinal).astype("int64")


In [50]:
df = pd.get_dummies(df, dtype=float)
df

,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,ExterQual,BsmtQual,...,SaleType_ConLw,SaleType_New,SaleType_Oth,SaleType_WD,SaleCondition_Abnorml,SaleCondition_AdjLand,SaleCondition_Alloca,SaleCondition_Family,SaleCondition_Normal,SaleCondition_Partial
0,60,65.0,8450,7,5,2003,2003,196.0,4,4,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1,20,80.0,9600,6,8,1976,1976,0.0,3,4,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
2,60,68.0,11250,7,5,2001,2002,162.0,4,4,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
3,70,60.0,9550,7,5,1915,1970,0.0,3,3,...,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0
4,60,84.0,14260,8,5,2000,2000,350.0,4,4,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1455,60,62.0,7917,6,5,1999,2000,0.0,3,4,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1456,20,85.0,13175,6,6,1978,1988,119.0,3,4,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1457,70,66.0,9042,7,9,1941,2006,0.0,5,3,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1458,20,68.0,9717,5,6,1950,1996,0.0,3,3,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0


In [51]:
df.isna().sum().sum()

np.int64(0)

In [52]:
correlation_matrix = df.corr()
correlation_matrix

,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,ExterQual,BsmtQual,...,SaleType_ConLw,SaleType_New,SaleType_Oth,SaleType_WD,SaleCondition_Abnorml,SaleCondition_AdjLand,SaleCondition_Alloca,SaleCondition_Family,SaleCondition_Normal,SaleCondition_Partial
MSSubClass,1.000000,-0.356718,-0.139781,0.032628,-0.059316,0.027850,0.040581,0.023573,0.016178,0.051122,...,0.014005,-0.045156,-0.014555,0.026359,0.005003,0.016241,0.030002,0.000983,0.024359,-0.051068
LotFrontage,-0.356718,1.000000,0.304522,0.234812,-0.053281,0.116685,0.083348,0.178469,0.165567,0.141836,...,-0.051283,0.128995,-0.023074,-0.091864,-0.021725,-0.036570,-0.018040,0.016250,-0.074146,0.127293
LotArea,-0.139781,0.304522,1.000000,0.105806,-0.005636,0.014228,0.013788,0.103321,0.055570,0.072336,...,-0.015040,0.020039,-0.005722,-0.002292,-0.029126,-0.013208,0.008966,-0.010781,0.005711,0.022635
OverallQual,0.032628,0.234812,0.105806,1.000000,-0.091932,0.572323,0.550684,0.407252,0.726278,0.629379,...,-0.021172,0.327412,-0.057962,-0.225013,-0.103535,-0.041677,-0.044950,-0.025515,-0.143282,0.323295
OverallCond,-0.059316,-0.053281,-0.005636,-0.091932,1.000000,-0.375983,0.073741,-0.125694,-0.138942,-0.164996,...,-0.019779,-0.156175,-0.050663,0.163684,-0.046367,-0.038888,-0.033444,-0.023873,0.161642,-0.151659
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SaleCondition_AdjLand,0.016241,-0.036570,-0.013208,-0.041677,-0.038888,-0.045601,-0.040294,-0.011783,-0.036145,-0.059166,...,-0.003073,-0.015827,-0.002378,0.020457,-0.014289,1.000000,-0.004772,-0.006177,-0.112080,-0.016038
SaleCondition_Alloca,0.030002,-0.018040,0.008966,-0.044950,-0.033444,-0.010104,-0.020727,-0.013748,-0.049563,-0.085444,...,-0.005337,-0.027489,-0.004131,0.035530,-0.024817,-0.004772,1.000000,-0.010729,-0.194663,-0.027856
SaleCondition_Family,0.000983,0.016250,-0.010781,-0.025515,-0.023873,-0.035785,-0.048056,-0.009535,-0.050478,-0.025427,...,-0.006909,-0.035587,-0.005348,0.028599,-0.032128,-0.006177,-0.010729,1.000000,-0.252006,-0.036062
SaleCondition_Normal,0.024359,-0.074146,0.005711,-0.143282,0.161642,-0.158427,-0.120577,-0.081539,-0.184302,-0.148452,...,0.027414,-0.645698,-0.097031,0.634322,-0.582947,-0.112080,-0.194663,-0.252006,1.000000,-0.654323


## Transformations Summarized

- Perform ordinal encoding on columns with "Qu" (quality) columns. 
- Convert Alley, Fence, and MasVnrType to boolean columns about whether they have values or are NA.